# Ensemble-FF-Fit - Ensemble Fitting of MACE

In this tutorial, we will demonstrate how we can use the [Ensemble-FF-Fit](https://github.com/Q-CAD/Ensemble-FF-Fit) and [MatEnsemble](https://github.com/Q-CAD/MatEnsemble) packages to fine-tune a [MACE](https://github.com/ACEsuit/mace) force field foundation model [MACE-OMAT](https://github.com/ACEsuit/mace-foundations). 

This example will fit force fields for bulk and 2D phases for Bi-Se heterostructures (InSe, NbSe2, and Al2O3 substrates), and is meant to be run on the [Perlmutter](https://docs.nersc.gov/systems/perlmutter/architecture) supercomputer. 

This notebook was written by Ryan Morelock.

## Outline of this notebook:

### Installation
1. **Build a Conda environment** supporting Ensemble-FF-Fit
2. **Create a Jupyter kernel** for this environment
3. **Write a helper script** to load necessary dependencies, and update the ```kernel.json``` file
4. **Check that the necessary executables** can be loaded

### A. Fitting an Ensemble of Force Fields on HPC
1. **Generate Structures** to be used as training and testing data, including 2D heterostructures
2. **Perform DFT calculations** on these modified structures using [vaspFlux](https://code.ornl.gov/rym/vaspflux)
3. **Parse MACE compatible input files** from DFT outputs using [parse2fit](https://code.ornl.gov/rym/parse2fit)
4. **Fit an ensemble of MACE force fields** using [MACE-freeze](https://github.com/7radians/mace-freeze)

### B. Uncertainty Quantification for Additional Fitting
1. **Run melting simulations** on the structures in training data
2. **Determine the images with the greatest uncertainty** along these trajectories using the force field ensemble
3. **Calculate DFT single-point energies** for the most uncertain, to be added as new training data
4. **Determine the best force field** from the ensemble based on DFT validation against AIMD structures

# Installation

## 1. Build a Conda environment

Ensemble-FF-Fit and its dependencies can be installed into a conda environment using the build scripts provided with the distribution. 

```git clone https://github.com/Q-CAD/Ensemble-FF-Fit/tree/main```

The conda environment can then be built by navigating into the ```Ensemble-FF-Fit``` directory and running

```. {chosen_build}.sh /desired/path/to/conda/environment```

## 2. Create a Jupyter kernel

A Jupyter kernel used to launch this notebook can be created after activating the built environment, for example:

```conda activate {your name}```

or your conda environment alias. Next, make the environment loadable as a Jupyter kernel:

```python -m ipykernel install --user --name {your name} --display-name "Your Name"```

## 3. Write a helper script

[MatEnsemble](https://github.com/Q-CAD/MatEnsemble) uses [Flux](https://flux-framework.readthedocs.io/en/latest/guides/learning_guide.html) to dynamically coordinate jobs on HPCs based on the resources available at runtime. 

Flux with MatEnsemble supported has currently been installed with [Spack](https://spack.io) on Perlmutter at:

```/global/cfs/cdirs/m5014/spack_py3.11/spack/share/spack/setup-env.sh```

We will need to load this environment, which contains its own Python, so that it can run alongside the Python installed into our Jupyter kernel. 

This can be accomplished via [helper-script](https://docs.nersc.gov/services/jupyter/how-to-guides/) formatting, as discussed in the Perlmutter documentation. An example helper script, which loads Cuda toolkit to support calculations on GPUs, as well as VASP module on Perlmutter, is provided below. 

In [1]:
helper_script = '''
#!/bin/bash
set -e

# Ensure module command exists
# source /etc/profile
# source /usr/share/Modules/init/bash

# Load Cray + CUDA stack
module load PrgEnv-gnu/8.5.0
module load cudatoolkit/12.4
module load craype-accel-nvidia80

# (Optional) Only load python module if you need it for tools
# Do NOT rely on it for the kernel Python
module load python/3.11
module load vasp/6.4.3-gpu

# Activate the spack
source /global/cfs/cdirs/m5014/spack_py3.11/spack/share/spack/setup-env.sh
spack load flux-sched
unset LUA_PATH LUA_CPATH

# Set the variables
CONDA_ENV="/global/homes/r/rym/.conda/envs/unified"
CONDA_SITE="$CONDA_ENV/lib/python3.11/site-packages"
CONDA_PYTHON="$CONDA_ENV/bin/python"

# Load the conda environment
conda activate $CONDA_ENV

# Add conda environment to PYTHONPATH before the Spack path
export PYTHONPATH="${CONDA_SITE}:${PYTHONPATH}"

exec "$@"
'''

This script must be made executable, using a command like: ```chmod u+x $HOME/.local/share/jupyter/kernels/env/kernel-helper.sh```

Finally, the ```kernel.json``` file will need to be updated so that the helper script is loaded. The script below forces the Python module that the Spack was compiled against to be the default, which is the current "hacky" way to get the MatEnsemble + Flux functionality.   

In [2]:
example_json = '''{
 "argv": [
  "{resource_dir}/kernel-helper.sh",
  "python", # Forces the loaded Python module to be used
  "-m",
  "ipykernel_launcher",
  "-f",
  "{connection_file}"
 ],
 "display_name": "Unified",
 "language": "python",
 "metadata": {
  "debugger": true
 }
}'''

## 4. Confirm the necessary executables

You should now be able to restart the notebook and load the kernel without issues (check the unfilled circle in upper right hand corner). 

We can also check that Flux is accessible based on our helper script that has been added to the location with the Jupyter Kernel. 

In [3]:
import sys, os, subprocess

print(sys.executable)
print(sys.version)

!which flux
!flux version

/global/cfs/cdirs/m5014/spack_py3.11/spack/opt/spack/linux-zen3/python-venv-1.0-52phvnnncpahv4yrmtaoif6xqifqfyez/bin/python
3.11.7 | packaged by conda-forge | (main, Dec 23 2023, 14:43:09) [GCC 12.3.0]
/global/cfs/cdirs/m5014/spack_py3.11/spack/opt/spack/linux-zen3/flux-core-0.73.0-5mhgz76sun6xxy6ybayeu55dut7gq325/bin/flux
commands:    		0.73.0
libflux-core:		0.73.0
build-options:		+systemd+hwloc.api==2.11.0+zmq==4.3.5


Finally, let's prepend the Ensemble-FF-Fit bin location to the ```PATH``` variable so we can use the Python executables provided; using these executables will make certain parts of the workflow much easier. 

In [4]:
import sys, os, shutil, subprocess, glob

In [5]:
def remove_path(path):
    if os.path.exists(path):
        if os.path.isdir(path):
            shutil.rmtree(path)
            return 
        os.remove(path)
    return 

Let's check that the ```mp_query``` executable is available to load, and if it is, all executables should be loadable.

In [6]:
env_bin = os.path.dirname(sys.executable)
os.environ["PATH"] = env_bin + os.pathsep + os.environ.get("PATH", "")

print("mp_query executable location:", shutil.which("mp_query"))

mp_query executable location: /global/homes/r/rym/.conda/envs/unified/bin/mp_query


# A. Fitting an Ensemble of Force Fields on HPC

## 1, 2. Generate Structures and Run DFT Calculations

We'll start by defining a couple of useful functions for file reading, writing and checking.

In [7]:
import yaml
import shutil
import json 

def write_json(path, dct):
    with open(path, 'w') as outfile:
        json.dump(dct, outfile, indent=4)

def write_yaml(path, dct):
    with open(path, 'w') as outfile:
        yaml.dump(dct, outfile, default_flow_style=False, sort_keys=False)
    return 

def write_string(path, string):
    with open(path, 'w') as outfile:
        outfile.write(string)

def read_file(path):
    with open(path, 'r') as infile:
        lst = infile.readlines()
    return lst

def copy_files(src_dir, dst_dir, patterns=()):
    os.makedirs(dst_dir, exist_ok=True)

    for fname in os.listdir(src_dir):
        if any(pat in fname for pat in patterns):
            src_path = os.path.join(src_dir, fname)
            dst_path = os.path.join(dst_dir, fname)

            if os.path.isfile(src_path):
                shutil.copy(src_path, dst_path)

def concatenate_files(filepaths, output_file):
    with open(output_file, "w") as outfile:
        for filepath in filepaths:
            with open(filepath, "r") as infile:
                # Read and write each file's content
                outfile.write(infile.read() + "\n")
    return 

def file_exists_anywhere(root_dir, filename):
    for _, _, files in os.walk(root_dir):
        if filename in files:
            return True
    return False

We can also write a function that we'll use to run executables from within the Notebook.

In [8]:
import subprocess

def MatEnsemble_submitter(cmd_args, working_directory):
    try:
        subprocess.run(
            cmd_args,
            cwd=working_directory,
            check=True,              # <-- THIS is the key
            stdout=subprocess.DEVNULL,
            stderr=subprocess.PIPE,  # capture error output
            text=True
        )
    except subprocess.CalledProcessError as e:
        if e.returncode == 1:
            print(
                "Workflow exited with code 1.\n"
                "This usually means no remaining jobs were found.\n"
                "Check whether the workflow has already converged."
            )
        else:
            print("Workflow failed with unexpected error:")
            print(e.stderr)

def dry_run_submitter(cmd_args, output_directory, dry_run=True):
    if dry_run is True:
        dry_args = cmd_args + ["--dry_run"]
        subprocess.run(dry_args) 
    else:
        run_args = ["srun", "flux", "start"] + cmd_args
        MatEnsemble_submitter(run_args, output_directory)

### a. Query Materials Project Bulk Structures

We will begin by creating a subdirectory for our VASP DFT calculations. 

In [9]:
dft_path = os.path.abspath(os.path.join(os.getcwd(), 'DFT'))
os.makedirs(dft_path, exist_ok=True)

Ensemble-FF-Fit has several command line executables installed that can be used to generate structures for DFT runs. These executables write and modify ```POSCAR``` files, which is the VASP default.

The ```mp_query``` executable pulls structures from the [Materials Project Database](https://next-gen.materialsproject.org) based on a provided YAML directive file. Let's write a YAML file that only queries experimentally-relevant DFT structures for this system, which are indicated by a star next to their entry on the MP website. 

**Note** For these examples, we will limit ourselves to structures with fewer than 40 atoms and energies above the hull of less than or equal to 100 meV/atom. This will help accelerate our DFT calculations. 

In [10]:
from mp_api.client import MPRester

def query_experimental_mpids(max_atoms, max_e_hull, chemsys):
    with MPRester('ky5EtinjFmpPqDQOO8a2Ept0RWdAOj59') as mpr:
        docs = mpr.materials.summary.search(
            theoretical=False,
            chemsys=chemsys,
            num_sites=(0, max_atoms),
            energy_above_hull=(0, max_e_hull),
            fields=["material_id"],
        )
    return [doc.material_id.string for doc in docs]

In [11]:
bi_se_mpids = query_experimental_mpids(40, 0.1, 'Bi-Se')
in_se_mpids = query_experimental_mpids(40, 0.04, 'In-Se')
nb_se_mpids = query_experimental_mpids(40, 0.001, 'Nb-Se')
al_o_mpids = query_experimental_mpids(40, 0.1, 'Al-O')

Retrieving SummaryDoc documents:   0%|          | 0/7 [00:00<?, ?it/s]

Retrieving SummaryDoc documents:   0%|          | 0/6 [00:00<?, ?it/s]

Retrieving SummaryDoc documents:   0%|          | 0/6 [00:00<?, ?it/s]

Retrieving SummaryDoc documents:   0%|          | 0/5 [00:00<?, ?it/s]

In [12]:
query_dct = {'mpids': bi_se_mpids + ['mp-568844'] + in_se_mpids + nb_se_mpids + al_o_mpids} # Add the BiSe rocksalt phase

query_yaml = os.path.join(dft_path, 'Heteros.yml')
write_yaml(query_yaml, query_dct)

We can now generate ```POSCAR``` files for our DFT runs. We will call this, and other, executables in this notebook using ```subprocess.run()```.

In [13]:
path_to_bulk_poscars = os.path.join(dft_path, 'structures/bulk')
os.makedirs(path_to_bulk_poscars, exist_ok=True)

#if not os.path.isdir(path_to_bulk_poscars):
subprocess.run(["mp_query",
                "--poscars_directory", path_to_bulk_poscars,
                "--max_atoms", "100",
                "--MP_yaml", query_yaml], stdout=subprocess.DEVNULL)

Retrieving SummaryDoc documents: 100%|██████████| 25/25 [00:00<00:00, 998643.81it/s]


CompletedProcess(args=['mp_query', '--poscars_directory', '/pscratch/sd/r/rym/Bi-Se_Heteros_MACE/DFT/structures/bulk', '--max_atoms', '100', '--MP_yaml', '/pscratch/sd/r/rym/Bi-Se_Heteros_MACE/DFT/Heteros.yml'], returncode=0)

These bulk structures provide a starting point for our fine-tuning dataset, but will need to be optimized with the DFT input parameters we'd like to use. Running this calculations will provide hands-on experience with MatEnsemble. 

### b. Optimizing Bulk Structures
Let's first create the YAML directive file and template submission script required by [vaspFlux](https://code.ornl.gov/rym/vaspflux) to create the inputs for VASP DFT jobs. We will set VASP parameters consistently across all calculations in this Notebook. 

In [14]:
input_path = os.path.join(dft_path, 'inputs_directory')
os.makedirs(input_path, exist_ok=True)

opt_dct = {'user_incar_settings': {'ALGO': "Normal",
                                   'EDIFF': 1.0e-05, 
                                   'EDIFFG': -0.01, 
                                   'ISIF': 3, 
                                   'NPAR': 16, 
                                   'KPAR': 4, 
                                   'ISMEAR': 0, 
                                   'SIGMA': 0.03, 
                                   'IVDW': 12,
                                   'NSW': 20, 
                                   'SYMPREC': 1.0e-06}, 
           'user_kpoints_settings': {'grid_density': 1000}}

opt_yaml_path = os.path.join(input_path, 'vdW_optimize.yml')
write_yaml(opt_yaml_path, opt_dct)           

vaspFlux's ```generate_vaspflux``` executable takes a template job submission file used to run individual VASP jobs on HPC. We provide an example template script, with GPU support, below.    

vaspFlux's ```submit_matsemble``` executable currently reads the number of individual tasks from this submission script, though there are plans to deprecate this behavior in the future.  

In [15]:
perlmutter_vasp_submission_string = '''#!/bin/bash
#
# Change to your account
# Also change in the srun command below
#SBATCH -A {ALLOCATION}
#
# Job naming stuff
#SBATCH -J {JOB_NAME}
#SBATCH -o %x-%j.out
#SBATCH -e %x-%j.err
#
# Requested time
#SBATCH -t {TIME}
#
# Requested queue
#SBATCH -C gpu
#SBATCH -q {PARTITION}
#
# Number of perlmutter nodes to use.
# Set the same value in the SBATCH line and NNODES
#SBATCH -N {NODES}

#SBATCH --exclusive
NNODES={NODES}

GPUS_PER_NODE={GPUS_PER_NODE}
GPUS_PER_TASK={GPUS_PER_TASK}
NGPUS=$(($GPUS_PER_NODE * $NNODES))

CPUS_PER_NODE={CPUS_PER_NODE}
CPUS_PER_TASK={CPUS_PER_TASK}
NCPUS=$(($CPUS_PER_NODE * $NNODES))

NUM_TASKS=$(($NGPUS / $GPUS_PER_TASK))

export OMP_NUM_THREADS=1
export OMP_PLACES=threads
export OMP_PROC_BIND=spread

module load vasp/6.4.3-gpu # Or version to be loaded\n

VASP_BINARY={VASP_EXECUTABLE}

# Run command. Modify to where ever you placed the binary and input files
srun -A m2113_g --ntasks=$NUM_TASKS --cpus-per-task $CPUS_PER_TASK --cpu-bind=cores --gpu-bind=none --gpus=$NGPUS $VASP_BINARY'''

perlmutter_vasp_submission_path = os.path.join(input_path, 'perlmutter_vasp_submission.sh')
write_string(perlmutter_vasp_submission_path, perlmutter_vasp_submission_string)

Now, let's call the ```generate_vaspflux``` executable to generate inputs for the queried `POSCAR` files. 

In [16]:
subprocess.run(["generate_vaspflux",
                    "--poscars_directory", path_to_bulk_poscars,
                    "--vasp_yaml", opt_yaml_path,
                    "--vasp_submission", perlmutter_vasp_submission_path, 
                    "--atoms_per_node", "80"], stdout=subprocess.DEVNULL)

CompletedProcess(args=['generate_vaspflux', '--poscars_directory', '/pscratch/sd/r/rym/Bi-Se_Heteros_MACE/DFT/structures/bulk', '--vasp_yaml', '/pscratch/sd/r/rym/Bi-Se_Heteros_MACE/DFT/inputs_directory/vdW_optimize.yml', '--vasp_submission', '/pscratch/sd/r/rym/Bi-Se_Heteros_MACE/DFT/inputs_directory/perlmutter_vasp_submission.sh', '--atoms_per_node', '80'], returncode=0)

We can now run ```submit_matsemble``` to submit and run these calculations. **Note:** most of the MatEnsemble-supported executables shown in this Notebook contain a `--dry_run` flag. This can help scope the number of calculations to be run, and what resources they require. 

In [17]:
dft_outputs_path = os.path.join(dft_path, 'outputs')
dft_bulk_outputs_path = os.path.join(dft_outputs_path, 'bulk')
os.makedirs(dft_bulk_outputs_path, exist_ok=True)

bulk_dft_cmd_args = ["matsemble_vaspflux",
                     "--parent_directory", 
                     path_to_bulk_poscars]

dry_run_submitter(bulk_dft_cmd_args, dft_bulk_outputs_path, dry_run=True)

No files to run; exiting workflow


### c. Generating defect structures
After finishing these bulk optimizations, we can copy the final structures to new directories and modify them to perform additional calculations. 

For example, let's create a `defect` directory and copy the `CONTCAR` files present in the `bulk` directory tree to `POSCAR` files in this new directory. This is conveniently performed with an the ```submit_vaspflux``` executable provided in vaspFlux. 

In [18]:
path_to_defect_poscars = os.path.join(dft_path, 'structures/defects')

if not os.path.isdir(path_to_defect_poscars):
    subprocess.run(["submit_vaspflux",
                "--parent_directory", path_to_bulk_poscars,
                "--move", "--move_to", path_to_defect_poscars], stdout=subprocess.DEVNULL)

Additionally, we can generate defective structures using the in-build ```generate_defects``` executable. Currently, we are passing the "-v" flag to generate only vacancies, but we encourage the user to explore the different possible defects that can be generated. 

**Note:** Larger structures can take a very long time to enumerate all defects (more unique defects to enumerate), so a conservative first pass is often the most prudent. 

In [19]:
if not file_exists_anywhere(path_to_defect_poscars, 'POSCAR.vasp'):
    subprocess.run(["generate_defects",
                "--poscars_directory", path_to_defect_poscars, 
                "-v", "-i"], stdout=subprocess.DEVNULL)

We won't want to run all of the defects that we've generated; so let's write a quick function to rename the ```POSCAR``` files so that we only run a single defect for each parent structure. 

In [20]:
from pathlib import Path 
import random

def rename_POSCARs(path, num_unique=1, name='unused_POSCAR'):
    paths_dct = {}
    
    for root, _, _ in os.walk(path):
        poscar_path = os.path.join(root, 'POSCAR')
        if os.path.exists(poscar_path):
            key = str(Path(poscar_path).parent.parent)
            if key not in list(paths_dct.keys()):
                paths_dct[key] = []
            paths_dct[key].append(poscar_path)

    sampled = []
    for path_key, path_lst in paths_dct.items():
        num_to_sample = max((0, len(path_lst)-num_unique))
        path_sampled = random.sample(path_lst, num_to_sample)
        sampled += path_sampled

    for sampled_path in sampled:
        unused_path = os.path.join(str(Path(sampled_path).parent), name)
        os.rename(sampled_path, unused_path)
            
    return

In [21]:
rename = False
if rename:
    rename_POSCARs(path_to_defect_poscars)

We will also write an input YAML for VASP single points, using many of the same input parameters as before.

In [22]:
from copy import deepcopy

sp_dct = deepcopy(opt_dct)
sp_dct['user_incar_settings']['NSW'] = 0
sp_dct['user_incar_settings']['ALGO'] = 'VeryFast' #'All'
sp_dct['user_incar_settings']['NELM'] = 250

sp_yaml_path = os.path.join(input_path, 'vdW_single_point.yml')
write_yaml(sp_yaml_path, sp_dct)

Now, let's generate the inputs and run the VASP workflow.

In [23]:
# Create the defect calculation inputs
subprocess.run(["generate_vaspflux",
                    "--poscars_directory", path_to_defect_poscars,
                    "--vasp_yaml", sp_yaml_path,
                    "--vasp_submission", perlmutter_vasp_submission_path, 
                    "--atoms_per_node", "80"], stdout=subprocess.DEVNULL)

submit_defects_MatEnsemble = False
dft_defects_outputs_path = os.path.join(dft_outputs_path, 'defects')
os.makedirs(dft_defects_outputs_path, exist_ok=True)

defects_dft_cmd_args = ["matsemble_vaspflux",
                     "--parent_directory", 
                     path_to_defect_poscars]

dry_run_submitter(defects_dft_cmd_args, dft_defects_outputs_path, dry_run=True)

No files to run; exiting workflow


### d. Rescaling structures 
We can run ```generate_EoS``` on the bulk structures to rescale them isotropically, preserving symmetries. 

In [24]:
path_to_rescaled_poscars = os.path.join(dft_path, 'structures/EoS')

if not os.path.isdir(path_to_rescaled_poscars):
    subprocess.run(["submit_vaspflux",
                "--parent_directory", path_to_bulk_poscars,
                "--move", "--move_to", path_to_rescaled_poscars], stdout=subprocess.DEVNULL)

if not file_exists_anywhere(path_to_rescaled_poscars, 'POSCAR.vasp'):
    subprocess.run(["generate_EoS",
                "--poscars_directory", path_to_rescaled_poscars, 
                "--total_displacements", "5", 
                "--displacement_spacing", "0.075"], stdout=subprocess.DEVNULL)

subprocess.run(["generate_vaspflux",
                    "--poscars_directory", path_to_rescaled_poscars,
                    "--vasp_yaml", sp_yaml_path,
                    "--vasp_submission", perlmutter_vasp_submission_path, 
                    "--atoms_per_node", "80"], stdout=subprocess.DEVNULL)

dft_EoS_outputs_path = os.path.join(dft_outputs_path, 'EoS')
os.makedirs(dft_EoS_outputs_path, exist_ok=True)

EoS_dft_cmd_args = ["matsemble_vaspflux",
                     "--parent_directory", 
                     path_to_rescaled_poscars]

dry_run_submitter(EoS_dft_cmd_args, dft_EoS_outputs_path, dry_run=True)

No files to run; exiting workflow


This and other functionality present in Ensemble-FF-Fit, but not shown here, provides simple ways to create a starting dataset with limited user intervention. 

### e. Isolated Atom Energies

We also need to generate isolated atomic energies at this level of theory for the MACE force field; we'll do that now. 

In [25]:
path_to_isolated_poscars = os.path.join(dft_path, 'structures/atomic')

elements = ['Bi', 'In', 'Nb', 'Se', 'Al', 'O']
paths_to_isolated_atoms_elements = [os.path.join(path_to_isolated_poscars, el) for el in elements]

from pymatgen.core.structure import Structure

lattice = [[20, 0, 0], [0, 20, 0], [0, 0, 20]]
coords = [[10, 10, 10]]

for i, element in enumerate(elements): 
    structure =  Structure(lattice=lattice, 
                           species=[element], 
                           coords=coords, 
                           coords_are_cartesian=True)
    os.makedirs(paths_to_isolated_atoms_elements[i], exist_ok=True)
    structure.to(os.path.join(paths_to_isolated_atoms_elements[i], 'POSCAR'), fmt='poscar')

In [26]:
subprocess.run(["generate_vaspflux",
                    "--poscars_directory", path_to_isolated_poscars,
                    "--vasp_yaml", sp_yaml_path,
                    "--vasp_submission", perlmutter_vasp_submission_path, 
                    "--atoms_per_node", "80"], stdout=subprocess.DEVNULL)

dft_atomic_outputs_path = os.path.join(dft_outputs_path, 'atomic')
os.makedirs(dft_atomic_outputs_path, exist_ok=True)

atomic_dft_cmd_args = ["matsemble_vaspflux",
                     "--parent_directory", 
                     path_to_isolated_poscars]

dry_run_submitter(atomic_dft_cmd_args, dft_atomic_outputs_path, dry_run=True)

No files to run; exiting workflow


We'll parse these isolated atom energies into a dictionary of {atomic_number: final_energy} for force-field fitting.

In [27]:
from pymatgen.io.vasp.outputs import Vasprun

def single_atom_energies(path):
    dct = {}
    for root, _, _ in os.walk(path):
        vasprun_path = os.path.join(root, 'vasprun.xml')
        if os.path.exists(vasprun_path):
            v = Vasprun(vasprun_path)
            number = int(v.structures[-1][0].specie.number)
            energy = float(v.final_energy)
            dct[number] = energy
    return dct

atomic_dct = single_atom_energies(path_to_isolated_poscars)

### f. 2D (Slabs + Heterostructures)

#### I. Slabs

We would like to fine-tune MACE-OMAT to 2D Bi-Se materials, including slabs and heterostructures. For now, we will limit ourselves to specific paths to converged structures / surface facets to create these  new structures; while this would not be too difficult of an extension to fully automate, we will only focus on a few important structures for our application. 

We will start by specifying dictionaries with film and substrate Material's Project IDs and constructing structure dictionaries, from which we'll build the heterostructures. 

In [28]:
# How many layers you would like for the film and the substrate 

film_layers_dct = {'mp-541837': '2', 'mp-27607': '2', 'mp-542615': '1', 
                   'mp-568844': '4', 'mp-23164': '1'}
substrate_layers_dct = {'mp-1143': '1', 'mp-2207': '2', 'mp-20485': '2'}

In [29]:
from pymatgen.core.structure import Structure

def get_VASP_structures_from_paths(path, layers_dct, replace_path='bulk', replace_with='slabs'):
    structure_dct = {}
    for root, _, _ in os.walk(path):
        for key, item in layers_dct.items():
            if key in root:
                vasprun_path = os.path.join(root, 'vasprun.xml')
                if os.path.exists(vasprun_path):
                    v = Vasprun(vasprun_path)
                    if v.converged:
                        s = v.structures[-1] # Last image
                        structure_dct[key] = {'path':  root.replace(replace_path, replace_with), 'layers': item, 'structure': s}                    
    return structure_dct

In [30]:
slabs_name = 'slabs'
path_to_slab_dir = os.path.join(dft_path, 'structures/slabs')

Let's next create a 0001 facet slab for 'mp-1143', which is the bulk alpha-Al2O3 phase and the one structure that isn't vdW-layered, the 001 surface facet for 'mp-568844', which is the rocksalt ordered cubic BiSe phase, and the 100 surface facet for 'mp-23164', which is the orthorhombic Bi2Se3 phase. 

We will update the dictionaries with versions of these slabs (for Al2O3, for example, we'll use the second slab generated after inverting the structure so that the oxygens are at the interface, and then reducing the size of the slab so that the heterostructures are smaller).

In [31]:
from pymatgen.core.surface import SlabGenerator
from pymatgen.io.ase import AseAtomsAdaptor
import numpy as np

film_bulk_dct = get_VASP_structures_from_paths(path_to_bulk_poscars, film_layers_dct, 
                                               replace_with=slabs_name)
substrate_bulk_dct = get_VASP_structures_from_paths(path_to_bulk_poscars, substrate_layers_dct, 
                                                   replace_with=slabs_name)

def set_vacuum(structure, vacuum_spacing, axis):
    aaa = AseAtomsAdaptor()
    atoms = structure.to_ase_atoms()
    atoms.center(vacuum=0.5 * vacuum_spacing, axis=2)
    new_structure = aaa.get_structure(atoms)
    return new_structure

def create_slab(structure, slab_index, facet=(0, 0, 1), min_slab_size=1, min_vacuum_size=2.4, invert=False):
    slabgen = SlabGenerator(structure, facet, min_slab_size=min_slab_size, 
                            min_vacuum_size=min_vacuum_size)
    slabs = slabgen.get_slabs()
    slab_to_use = slabs[slab_index]

    if invert:
        arr = np.array([[0, 0, 1] for f in range(len(slab_to_use.frac_coords))])
        new_frac_coords = np.subtract(arr, slab_to_use.frac_coords)
        slab_to_use = Structure(slab_to_use.lattice, 
                            slab_to_use.species, 
                            new_frac_coords, to_unit_cell=True)
        
    return slab_to_use

vacuum_size = 2.4

# Al2O3 slab
al2o3_001_inverted = create_slab(substrate_bulk_dct['mp-1143']['structure'], 1, invert=True)
al2o3_remove = [i for i in range(len(al2o3_001_inverted)) if al2o3_001_inverted[i].coords[2] < 19.5]
al2o3_001_inverted.remove_sites(al2o3_remove)
al2o3_001_inverted = set_vacuum(al2o3_001_inverted, vacuum_size, axis=2)
substrate_bulk_dct['mp-1143']['structure'] = al2o3_001_inverted

# BiSe slab
bise_001_2_layer = create_slab(film_bulk_dct['mp-568844']['structure'], facet=(0, 1, 0),
                               slab_index=0, min_slab_size=8)
bise_001_2_layer = set_vacuum(bise_001_2_layer, vacuum_size, axis=2)
film_bulk_dct['mp-568844']['structure'] = bise_001_2_layer

# Bi2Se3 ortho slab
bi2se3_ortho_1_layer = create_slab(film_bulk_dct['mp-23164']['structure'], facet=(1, 0, 0), 
                                   slab_index=0, min_slab_size=1)
bi2se3_ortho_1_layer = set_vacuum(bi2se3_ortho_1_layer, vacuum_size, axis=2)
film_bulk_dct['mp-23164']['structure'] = bi2se3_ortho_1_layer

#print(bi2se3_ortho_1_layer.to(fmt='poscar'))

We'll create the slabs to be run with a custom slab generation code. 

In [32]:
slab_code = r"""
from pymatgen.core.structure import Structure
from vdW_structures.vdW_structure import VdWStructure
import sys

structure = Structure.from_str(sys.argv[1], fmt='poscar')
minimum_vdW_gap = eval(sys.argv[2])
vacuum = eval(sys.argv[3])
num_layers = eval(sys.argv[4])

def resize_list(n, arr):
    return (arr * ((n + len(arr) - 1) // len(arr)))[:n]

vdW_structure = VdWStructure(structure, minimum_vdW_gap=minimum_vdW_gap)
structure_layers = resize_list(num_layers, [i for i in range(len(vdW_structure.vdW_layers))])
vdW_structure = vdW_structure.extend_vdW_layers(structure_layers)
vdW_structure = vdW_structure.set_vacuum(vacuum)

print(vdW_structure.structure.to(fmt='poscar'))
"""

def slabs_from_dct(dct, slab_code, vacuum=10, minimum_vdW_gap=2.2):
    for slab_key, slab_dct in dct.items():
        slab_out = subprocess.check_output(
                    [
                        "/global/homes/r/rym/.conda/envs/unified/bin/python",
                        "-c",
                        slab_code, slab_dct['structure'].to(fmt='poscar'), 
                        str(minimum_vdW_gap), str(vacuum), str(slab_dct['layers'])],
                    text=True
                )
        slab = Structure.from_str(slab_out, fmt='poscar')
        slab_dir = os.path.join(str(slab_dct['path']), str(slab_dct['layers']))
        os.makedirs(slab_dir, exist_ok=True)
        slab.to(os.path.join(slab_dir, 'POSCAR'), fmt='poscar')                                         
    return 

if not os.path.exists(path_to_slab_dir):
    slabs_from_dct(film_bulk_dct, slab_code)
    slabs_from_dct(substrate_bulk_dct, slab_code)

We can iterate through the directory containing the slabs and call an executable included in Ensemble-FF-Fit that rescales gap distances using the HeteroBuilder package.  

In [33]:
def get_gap_index(substrate_index, hetero):
    if hetero:
        return int(substrate_index)
    else:
        return int(substrate_index) - 1 # within substrate layer

def rescale_VdW_gaps(path_to_gapped_structures, substrate_dct, 
                     displacement_spacing=0.2, total_displacements=5, 
                     minimum_vdW_gap=2.2, hetero=False):
    
    for root, _, _ in os.walk(path_to_gapped_structures):
        poscar_path = os.path.join(root, 'POSCAR')
        if os.path.exists(poscar_path):
            substrate_key = None
            for key in list(substrate_dct.keys()):
                if key in poscar_path:
                    substrate_key = key
                    use_index = get_gap_index(substrate_dct[substrate_key]['layers'], hetero)
                    if use_index == 0: # No intergap distance to be exfoliated
                        pass
                    else:
                        subprocess.run(["generate_vdW_spacing",
                            "--poscars_directory", root,
                            "--displacement_spacing", str(displacement_spacing),
                            "--total_displacements", str(total_displacements),
                            "--displace_layer_indices", str(use_index),
                            "--layer_tolerance", str(minimum_vdW_gap - 0.0001)], 
                                   stdout=subprocess.DEVNULL)
    return 

if not os.path.exists(path_to_slab_dir):
    rescale_VdW_gaps(path_to_slab_dir, substrate_bulk_dct, hetero=False)
    rescale_VdW_gaps(path_to_slab_dir, film_bulk_dct, hetero=False)

We can now run DFT calculations on the generated slabs. 

In [34]:
subprocess.run(["generate_vaspflux",
                    "--poscars_directory", path_to_slab_dir,
                    "--vasp_yaml", sp_yaml_path,
                    "--vasp_submission", perlmutter_vasp_submission_path, 
                    "--atoms_per_node", "50"], stdout=subprocess.DEVNULL)

dft_slabs_outputs_path = os.path.join(dft_outputs_path, 'slabs')
os.makedirs(dft_slabs_outputs_path, exist_ok=True)

slab_dft_cmd_args = ["matsemble_vaspflux",
                     "--parent_directory", 
                     path_to_slab_dir]

dry_run_submitter(slab_dft_cmd_args, dft_slabs_outputs_path, dry_run=True)

No files to run; exiting workflow


#### II. Exfoliated Heterostructures

Now, let's create heterostructures from the slabs using a similar custom function and write them to a directory for DFT runs. 

**Note:** This is a little more hands-on, but we will create heterointerfaces by combining structures from the substrate and film dictionaries. We will also set an atom range on the size of the heterostructures we'd like to consider. 

In [35]:
heterostructure_code = r"""
from vdW_structures.vdW_structure import VdWStructure
from vdW_structures.vdW_heterostructure import VdWHeterostructureGenerator
from pymatgen.analysis.interfaces.zsl import ZSLGenerator
from pymatgen.core.structure import Structure
import sys

film = Structure.from_str(sys.argv[1], fmt='poscar')
substrate = Structure.from_str(sys.argv[2], fmt='poscar')

minimum_vdW_gap = eval(sys.argv[3])

max_area_ratio_tol = eval(sys.argv[4])
max_area = eval(sys.argv[5])
max_length_tol = eval(sys.argv[6])
max_angle_tol = eval(sys.argv[7])
vacuum_over_film = eval(sys.argv[8])

num_film_layers = eval(sys.argv[9])
num_substrate_layers = eval(sys.argv[10])

min_atoms = eval(sys.argv[11])
max_atoms = eval(sys.argv[12])

def resize_list(n, arr):
    return (arr * ((n + len(arr) - 1) // len(arr)))[:n]

vdW_substrate = VdWStructure(substrate, minimum_vdW_gap=minimum_vdW_gap)
substrate_layers = resize_list(num_substrate_layers, [i for i in range(len(vdW_substrate.vdW_layers))])
vdW_substrate = vdW_substrate.extend_vdW_layers(substrate_layers)

vdW_film = VdWStructure(film, minimum_vdW_gap=minimum_vdW_gap)
film_layers = resize_list(num_film_layers, [i for i in range(len(vdW_film.vdW_layers))])
vdW_film = vdW_film.extend_vdW_layers(film_layers)

zsl = ZSLGenerator(max_area_ratio_tol = max_area_ratio_tol, 
                   max_area = max_area,
                   max_length_tol = max_length_tol,
                   max_angle_tol = max_angle_tol)

try:
    structs, props = VdWHeterostructureGenerator().generate_vdW_heterostructures(film=vdW_film, 
                                                                             substrate=vdW_substrate,
                                                                             zsl_generator=zsl,
                                                                             vacuum_over_film=vacuum_over_film, 
                                                                             get_unique=False)

    use_structs = [s for s in structs if len(s.structure) > min_atoms and len(s.structure) < max_atoms]
    print(use_structs[0].structure.to(fmt='poscar'))
except (ValueError, IndexError): 
    print('No structure')
"""

path_to_heteros = os.path.join(dft_path, 'structures/heteros')

def heteros_from_dcts(write_path, film_dct, substrate_dct, vacuum=10, minimum_vdW_gap=2.2, 
                     max_area_ratio_tol=0.15, max_area=250, max_length_tol=0.15, max_angle_tol=0.15, 
                     min_atoms=0, max_atoms=300):

    for film_key, film_layer in film_dct.items():
        for substrate_key, substrate_layer in substrate_dct.items():
            film_num_layers = str(film_layer['layers'])
            substrate_num_layers = str(substrate_layer['layers'])
            hetero_out = subprocess.check_output(
                    [
                        "/global/homes/r/rym/.conda/envs/unified/bin/python",
                        "-c",
                        heterostructure_code,
                        film_layer['structure'].to(fmt='poscar'),
                        substrate_layer['structure'].to(fmt='poscar'),
                        str(minimum_vdW_gap-0.0001), str(max_area_ratio_tol), 
                        str(max_area), str(max_length_tol), str(max_angle_tol), 
                        str(vacuum), film_num_layers, substrate_num_layers, 
                        str(min_atoms), str(max_atoms)
                    ],
                    text=True
                )

            try:
                heterostructure = Structure.from_str(hetero_out, fmt='poscar')
                hetero_path = os.path.join(write_path, substrate_key, substrate_num_layers, film_key, film_num_layers) #, str(len(heterostructure)))
                os.makedirs(hetero_path, exist_ok=True)
                heterostructure.to(os.path.join(hetero_path, 'POSCAR'), fmt='poscar')
                print(f'{film_key} {substrate_key} {heterostructure.composition} {len(heterostructure)}')
            except (IndexError, TypeError):
                print(f'{film_key} {substrate_key} failed to find heterostructure within {min_atoms} {max_atoms}')
    return 

path_to_exfoliated_heteros = os.path.join(path_to_heteros, 'exfoliated')
if not os.path.isdir(path_to_exfoliated_heteros):
    heteros_from_dcts(path_to_exfoliated_heteros, film_bulk_dct, 
                   substrate_bulk_dct, min_atoms=0, max_atoms=80)

**Note:** Not all film/substrate combinations will construct and return a valid heterointerface, but that is ok. We will expand our heterostructure sizes for the UQ with MD. 

We can now create exfoliation curves at the heterointerface from these generated heterostructures by specifying which layers constitute the film and the substrate. 

In [36]:
if not os.path.isdir(path_to_exfoliated_heteros):
    rescale_VdW_gaps(path_to_exfoliated_heteros, substrate_bulk_dct, 
                     displacement_spacing=0.2, total_displacements=5, 
                     minimum_vdW_gap=2.2, hetero=True)

Now, let's submit single point calculations for these with MatEnsemble. 

In [37]:
subprocess.run(["generate_vaspflux",
                    "--poscars_directory", path_to_exfoliated_heteros,
                    "--vasp_yaml", sp_yaml_path,
                    "--vasp_submission", perlmutter_vasp_submission_path, 
                    "--atoms_per_node", "80"], stdout=subprocess.DEVNULL)

dft_exfol_outputs_path = os.path.join(dft_outputs_path, 'heteros/exfoliated')
os.makedirs(dft_exfol_outputs_path, exist_ok=True)

exfol_dft_cmd_args = ["matsemble_vaspflux",
                     "--parent_directory", 
                     path_to_exfoliated_heteros]

dry_run_submitter(exfol_dft_cmd_args, dft_exfol_outputs_path, dry_run=True)

<string>:31: BadInputSetWarning: Overriding the POTCAR functional is generally not recommended  as it significantly affects the results of calculations and compatibility with other calculations done with the same input set. Note that some POTCAR symbols specified in the configuration file may not be available in the selected functional.


Printing all VASP directories to run...

Task ID: 0, Path: /pscratch/sd/r/rym/Bi-Se_Heteros_MACE/DFT/structures/heteros/exfoliated/mp-1143/1/mp-542615/1/1/spacing_0.6/POSCAR, CPUs: 64, GPUs: 4

Total CPUs = 64, Total GPUs = 4. Do not forget to request 1 additional node for resource management!


#### III. MD Heterostructures

We can also increase the size of our heterostructures and use them to run MD, sampling new structures to add back in during UQ. Let's create the heterostructures as before, but this time expand the sizes we allow. 

**Note:** This can change the twist angle and registry between the films and substrates, allowing us to get different structures than what were used for the exfoliation curves. 

In [38]:
path_to_md_heteros = os.path.join(path_to_heteros, 'MD')
if not os.path.isdir(path_to_md_heteros):
    heteros_from_dcts(path_to_md_heteros, film_bulk_dct, 
                   substrate_bulk_dct, min_atoms=80, max_atoms=160)

We will perform single point calculations on these as well. 

**Note:** some of these will be used as starting structures for our AIMD validation data. 

In [ ]:
subprocess.run(["generate_vaspflux",
                    "--poscars_directory", path_to_md_heteros,
                    "--vasp_yaml", sp_yaml_path,
                    "--vasp_submission", perlmutter_vasp_submission_path, 
                    "--atoms_per_node", "50"], stdout=subprocess.DEVNULL)

dft_MD_hetero_outputs_path = os.path.join(dft_outputs_path, 'heteros/MD')
os.makedirs(dft_MD_hetero_outputs_path, exist_ok=True)

MD_hetero_dft_cmd_args = ["matsemble_vaspflux",
                     "--parent_directory", 
                     path_to_md_heteros]

dry_run_submitter(MD_hetero_dft_cmd_args, dft_MD_hetero_outputs_path, dry_run=True)

### g. AIMD Validation Runs

We will also validate our force fields on AIMD thermalizations of heterostructures with Bi2Se3 ```mp-541837``` as the film. We will 1) run a short ion optimization of the starting structures, 2) perform AIMD melts of these systems at temperature and 3) perform single point calculations on each image from the AIMD, which will constitute unseen validation data. 

Let's start by setting up and performing a simultaneous cell and ion optimization. This will give use "unique" starting structures for the validation, which will not be included in the training data (as UQ, for instance). 

In [ ]:
def get_paths_by_pattern(path, pattern):
    paths = []
    for root, _, _ in os.walk(path):
        if pattern in root:
            try:
                vasprun = Vasprun(os.path.join(root, 'vasprun.xml'))
                if vasprun.converged:
                    paths.append(root)  
            except FileNotFoundError:
                pass
    return paths

mp_541837_paths = get_paths_by_pattern(path_to_md_heteros, 'mp-541837')

path_to_aimd_heteros = os.path.join(dft_path, 'structures/AIMD')
path_to_aimd_runs = os.path.join(path_to_aimd_heteros, 'runs')
path_to_aimd_sps = os.path.join(path_to_aimd_heteros, 'single_points')

os.makedirs(path_to_aimd_runs, exist_ok=True)
os.makedirs(path_to_aimd_sps, exist_ok=True)

if not os.path.isdir(path_to_aimd_runs):
    for path in mp_541837_paths:
        subprocess.run(["submit_vaspflux",
                "--parent_directory", path,
                "--move", "--move_to", path_to_aimd_runs], 
                stdout=subprocess.DEVNULL)

In [ ]:
opt_dct['user_incar_settings']['NSW'] = 30
opt_dct['user_incar_settings']['ALGO'] = 'VeryFast'
opt_dct['user_incar_settings']['NELM'] = 80

write_yaml(opt_yaml_path, opt_dct)

In [ ]:
subprocess.run(["generate_vaspflux",
                    "--poscars_directory", os.path.join(path_to_aimd_runs, 'MD'),
                    "--vasp_yaml", opt_yaml_path,
                    "--vasp_submission", perlmutter_vasp_submission_path, 
                    "--atoms_per_node", "100"], stdout=subprocess.DEVNULL)

dft_AIMD_runs_outputs_path = os.path.join(dft_outputs_path, 'AIMD/runs')
os.makedirs(dft_AIMD_runs_outputs_path, exist_ok=True)

AIMD_runs_dft_cmd_args = ["matsemble_vaspflux",
                     "--parent_directory", 
                     os.path.join(path_to_aimd_runs, 'MD')]

dry_run_submitter(AIMD_runs_dft_cmd_args, dft_AIMD_runs_outputs_path, dry_run=True)

We will next create an input YAML file for AIMD runs at temperature. 

**Note:** This protocol is meant to melt/mix the crystalline heterostructures that have been ionically optimized. We will run single points on select images to make them compatible with the rest of the training/validation data, so it is not important that (for example) the ```ENCUT``` tag is lower than the default. 

In [ ]:
from copy import deepcopy

aimd_dct = deepcopy(opt_dct)
aimd_dct['user_incar_settings']['IBRION'] = 0
aimd_dct['user_incar_settings']['ENCUT'] = 400
aimd_dct['user_incar_settings']['POTIM'] = 3.0
aimd_dct['user_incar_settings']['TEBEG'] = 2000
aimd_dct['user_incar_settings']['TEEND'] = 2000
aimd_dct['user_incar_settings']['MDALGO'] = 2
aimd_dct['user_incar_settings']['SMASS'] = 1.0
aimd_dct['user_incar_settings']['ISYM'] = 0
aimd_dct['user_incar_settings']['ISIF'] = 2
aimd_dct['user_incar_settings']['NSW'] = 100
aimd_dct['user_incar_settings']['PREC'] = 'Low'

aimd_yaml_path = os.path.join(input_path, 'vdW_AIMD.yml')
write_yaml(aimd_yaml_path, aimd_dct)

In [ ]:
subprocess.run(["generate_vaspflux",
                    "--poscars_directory", os.path.join(path_to_aimd_runs, 'MD'),
                    "--vasp_yaml", aimd_yaml_path,
                    "--vasp_submission", perlmutter_vasp_submission_path, 
                    "--atoms_per_node", "100"], stdout=subprocess.DEVNULL)

dft_AIMD_runs_outputs_path = os.path.join(dft_outputs_path, 'AIMD/runs')
os.makedirs(dft_AIMD_runs_outputs_path, exist_ok=True)

AIMD_runs_dft_cmd_args = ["matsemble_vaspflux",
                     "--parent_directory", 
                     os.path.join(path_to_aimd_runs, 'MD')]

dry_run_submitter(AIMD_runs_dft_cmd_args, dft_AIMD_runs_outputs_path, dry_run=True)

Once these have finished, we can 1) select the images we'd like to use and 2) shuttle them to the ```single_points``` directory, where we will run DFT single points on the selected images. 

**Note:** We will pull the images from the `XDATCAR` files instead of the `vasprun.xml` file, and sample the images from here.

In [ ]:
from pymatgen.io.vasp.outputs import Xdatcar

def get_write_path(input_path, output_path):
    p1 = Path(input_path).resolve()
    p2 = Path(output_path).resolve()

    common_prefix_str = os.path.commonpath([str(p1), str(p2)])
    common_path = Path(common_prefix_str)

    p1_parts = list(p1.parts)
    p2_parts = list(p2.parts)
    common_parts = list(common_path.parts)
    
    divergence_index = len(common_parts)
    
    unique_p1 = Path(*p1_parts[divergence_index:])
    unique_p2 = Path(*p2_parts[divergence_index:])

    new_p = Path(output_path, *unique_p1.parts[1:])

    return str(new_p)

def sample_lst(lst, num_samples):
    indices = np.round(np.linspace(0, len(lst) - 1, num_samples)).astype(int)
    return indices, [lst[i] for i in indices]

def sample_from_Xdatcars(in_path, out_path, num_samples=20):
    for root, _, _ in os.walk(in_path):
        xdatcar_path = os.path.join(root, 'XDATCAR')
        if os.path.exists(xdatcar_path):
            xd = Xdatcar(xdatcar_path)
            images, sampled_structures = sample_lst(xd.structures, num_samples)
            write_path = get_write_path(root, out_path)
            for i, sampled in enumerate(sampled_structures):
                image_path = os.path.join(write_path, str(images[i]))
                os.makedirs(image_path, exist_ok=True)
                sampled.to(filename=os.path.join(image_path, 'POSCAR'), fmt='poscar')
    return 

if not os.listdir(path_to_aimd_sps):
    sample_from_Xdatcars(path_to_aimd_runs, path_to_aimd_sps)

In [ ]:
subprocess.run(["generate_vaspflux",
                    "--poscars_directory", path_to_aimd_sps,
                    "--vasp_yaml", sp_yaml_path,
                    "--vasp_submission", perlmutter_vasp_submission_path, 
                    "--atoms_per_node", "100"], stdout=subprocess.DEVNULL)

sp_AIMD_runs_outputs_path = os.path.join(dft_outputs_path, 'AIMD/single_points')
os.makedirs(sp_AIMD_runs_outputs_path, exist_ok=True)

AIMD_runs_sp_cmd_args = ["matsemble_vaspflux",
                     "--parent_directory", 
                     path_to_aimd_sps]

dry_run_submitter(AIMD_runs_sp_cmd_args, sp_AIMD_runs_outputs_path, dry_run=True)

### h. Uncertainty Quantification

Finally, let's create a directory where our uncertainty quantification runs will go. 

In [ ]:
path_to_uq_poscars = os.path.join(dft_path, 'structures/UQ')
os.makedirs(path_to_uq_poscars, exist_ok=True)

## 3. Parse DFT Calculations

Here, we will demonstrate the ```parse2fit``` function call that will allow us to parse the raw VASP outputs into an extended .xyz file for MACE force field fitting. 

We will start by writing a generic YAML file to guide the parsing (see ```parse2fit``` documentation for formatting). **Note**: Don't worry about the errors with weighting that are output; parse2fit was originally designed for more complicated ReaxFF data formatting. For MACE, we just need  extended.xyz files. 

In [ ]:
from glob import glob
from pathlib import Path
import re

ff_path = os.path.join(os.getcwd(), 'FF')

def get_generation(check_path, pattern):
    pattern_path = os.path.join(check_path, pattern)
    generations = glob(pattern_path)
    generation_names = [Path(g).name for g in generations]
    if generation_names:
        sorted_generations = sorted(generations)
        generation = str(re.findall(r'\d+', sorted_generations[-1])[0])
    else:
        generation = '1'
    return generation

generation = get_generation(ff_path, 'generation_*/')
print(f'Generation {generation}')

In [ ]:
# Get paths and labels for the unique generations
uq_generations_paths = glob(os.path.join(path_to_uq_poscars, "generation_*/"))
uq_generations = [re.search(r'\d+', Path(s).name).group() for s in uq_generations_paths]
uq_generations_labels = [f'UQ{g}' for g in uq_generations]

In [ ]:
data_directory = os.path.abspath(os.path.join(os.getcwd(), 'Data'))
parsed_data_directory = os.path.join(data_directory, f'generation_{generation}')
os.makedirs(data_directory, exist_ok=True)

eos_label_name = 'EoS'
defect_label_name = 'Defect'
slab_label_name = 'Slab'
hetero_label_name = 'Exfoliated'
AIMD_label_name = 'AIMD'

label_path_dct = {eos_label_name: path_to_rescaled_poscars, 
                 defect_label_name: path_to_defect_poscars, 
                 slab_label_name: path_to_slab_dir, 
                 hetero_label_name: path_to_exfoliated_heteros, 
                 AIMD_label_name: path_to_aimd_sps}

parse2fit_string = f"""# extended .xyz Layout

output_format: 'fitsnap' # consistent tag across all .yaml

generation_parameters:
  runs_to_generate: 1
  output_directory: {parsed_data_directory}

method_parameters:
  scraper:
    scraper: 'xyz'
  path:
    dataPath: 'XYZ'

input_paths:
  # Set a path here
  #
  {eos_label_name}:
    directories:
      - {label_path_dct[eos_label_name]}
    energy: True
    forces: True
    lattice_vectors: True
    stress_vectors: True
    
  {defect_label_name}:
    directories:
      - {label_path_dct[defect_label_name]}
    energy: True
    forces: True
    lattice_vectors: True
    stress_vectors: True
    
  {slab_label_name}:
    directories:
      - {label_path_dct[slab_label_name]}
    energy: True
    forces: True
    lattice_vectors: True
    stress_vectors: True
    
  {hetero_label_name}:
    directories:
      - {label_path_dct[hetero_label_name]}
    energy: True
    forces: True
    lattice_vectors: True
    stress_vectors: True
    
  {AIMD_label_name}:
    directories:
      - {label_path_dct[AIMD_label_name]}
    energy: True
    forces: True
    lattice_vectors: True
    stress_vectors: True"""

for i, label in enumerate(uq_generations_labels):
    append_string =   f"""  {label}:
    directories:
      - {uq_generations_paths[i]}
    energy: True
    forces: True
    lattice_vectors: True
    stress_vectors: True"""
    
    parse2fit_string = parse2fit_string + '\n\n' + append_string
    

parse2fit_path = os.path.join(data_directory, 'p2f_example.yml')
write_string(parse2fit_path, parse2fit_string)

Let's set a variable to the path of the .xyz file to write and delete it if it already exists. Then, let's run ```generate_parse2fit``` on these VASP directories to create example training data from the VASP input files. 

In [ ]:
xyz_write_path = Path(os.path.join(parsed_data_directory, 'fitsnap_run_0/XYZ/'))
xyz_paths = [str(p) for p in list(xyz_write_path.glob('*.xyz'))]
if os.path.isdir(xyz_write_path):
    pass
else:
    subprocess.run(["generate_parse2fit", parse2fit_path])
    xyz_paths = [str(p) for p in list(xyz_write_path.glob('*.xyz'))]

The file should show structures run with DFT and parsed for fitting, which we can quickly check with our ```read_file``` function.

In [ ]:
#print(xyz_paths)
read_file(xyz_paths[0])[0:5]

We now have data formatted for MACE force field fitting! 

By changing which data is passed in the input YAML directive, parse2fit can be used to generate different combinations of training data, further influencing the ensemble force field fit. 

Varying the training data in this manner can be useful to quickly assess (1) which data is most beneficial for training, and (2) how variable the resulting force fields become when subsets of the full training set are used. 

## 4. Fit an Ensemble of Force Fields

So far, we've explored different ways to generate structures for MACE force field fitting and evaluate their properties with DFT, most of which are near-equilibrium. This limited data is likely insufficient to reliably fine-tune a foundation model for finite-temperature MD. 

To fit the following force field ensemble, we will start from combinations of our parsed DFT training data, constructing training and testing sets. As we perform UQ over multiple generations with ```Ensemble-FF-Fit```, we will quantify how performance changes as new data is added. 

In [ ]:
ff_generation_path = os.path.join(ff_path, f'generation_{generation}')
ff_inputs_path = os.path.join(ff_generation_path, 'inputs_directory')
os.makedirs(ff_inputs_path, exist_ok=True)

In [ ]:
from itertools import combinations

def concatenate_and_write(train_filepaths, test_filepaths, output_dir, 
                          train_name, test_name, extension='.xyz', write=True):
    full_output_directories = []
    for files in train_filepaths:
        out_dir = "_".join([Path(path).name.strip(extension) for path in files])
        full_out = os.path.join(output_dir, out_dir)
        os.makedirs(full_out, exist_ok=True)
        train_out, test_out = os.path.join(full_out, train_name), os.path.join(full_out, test_name)
        if write == True:
            print(f'Writing to {full_out}')
            concatenate_files(files, train_out)
            concatenate_files(test_filepaths, test_out)
        full_output_directories.append(full_out)
    return full_output_directories

def get_data_combos(xyz_dct, labels=[], always_include_labels=[]):
    combo_paths = [xyz_dct[label] for label in labels if label not in always_include_labels and label in list(xyz_dct.keys())]
    include_paths = [xyz_dct[label] for label in always_include_labels if label in list(xyz_dct.keys())]
    
    combos = []
    for i in range(1, len(combo_paths)+1):
        combos += combinations(combo_paths, r=i)
    if not combos:
        combo_list = include_paths
    else:
        combo_list = [c + tuple(include_paths) for c in combos]
    return combo_list

def combine_and_write(path_to_xyzs, output_dir, train_labels=[], test_labels=[], 
                      always_in_training=[], always_in_testing=[], 
                      train_name='train.xyz', test_name='test.xyz', write=True):
    
    xyz_write_path = Path(path_to_xyzs)
    xyz_paths = [str(p) for p in list(xyz_write_path.glob('*.xyz'))]

    xyz_dct = {}
    for label in train_labels + test_labels:
        try:
            label_path = [path for path in xyz_paths if label in path][0]
            xyz_dct[label] = label_path
        except IndexError:
            pass

    train_combos = get_data_combos(xyz_dct, train_labels, always_in_training)
    test_combos = get_data_combos(xyz_dct, test_labels, always_in_testing)

    output_dirs = concatenate_and_write(train_combos, test_combos, output_dir, 
                         train_name, test_name, write=write)
    
    return output_dirs

if not os.listdir(ff_inputs_path):
    write_combos = True
else:
    write_combos = False

ff_output_dirs = combine_and_write(xyz_write_path, ff_inputs_path, 
                  train_labels=[eos_label_name, defect_label_name, slab_label_name, hetero_label_name] + uq_generations_labels, 
                  test_labels=[AIMD_label_name],
                  always_in_training=[eos_label_name], # + uq_generations_labels, 
                  always_in_testing=[AIMD_label_name], write=write_combos)

Now that we have the data, our next step will be to create the input files that dictate how the force field is fit. 

We can use a generic dictionary template for this fitting, and write a simple function which reads this template, makes modifications, and writes out ```config.yml``` files to subdirectories located in the ```inputs_directory```. 

In [ ]:
freeze = [6, 7]
weights = [(1.0, 20.0, 100.0), (1.0, 20.0, 50.0), (1.0, 20.0, 20.0)]

base_MACE_dct = {'energy_key': 'energy', 
                 'forces_key': 'forces', 
                 'stress_key': 'stress', 
                 'device': 'cuda', 
                 'batch_size': 10, 
                 'max_num_epochs': 100,
                 'num_samples_pt': 300,
                 'swa': False,
                 'model': 'MACE',
                 'stress_weight': 1.0,
                 'forces_weight': 20.0,
                 'energy_weight': 100.0,
                 'freeze': 6,
                 'E0s': atomic_dct,
                 'multiheads_finetuning': False,
                 'valid_fraction': 0.15,
                 'enable_cueq': False}

def write_MACE_YAMLs(default_FF_yaml, write_path, freeze, weights):
    for f in freeze:
        f_dir = os.path.join(write_path, f'freeze_{str(f)}')
        #os.makedirs(f_dir, exist_ok=True)
        for i, weight in enumerate(weights):
            w_dir = os.path.join(f_dir, f'energy_{str(i+1)}')
            os.makedirs(w_dir, exist_ok=True)
            base_MACE_dct['freeze'] = f
            base_MACE_dct.update({'stress_weight': weight[0], 'forces_weight': weight[1], 'energy_weight': weight[2]})
            write_yaml(os.path.join(w_dir, 'config.yml'), base_MACE_dct)
    return 

for ff_output_dir in ff_output_dirs: 
    write_MACE_YAMLs(base_MACE_dct, ff_output_dir, freeze, weights)

Let's next pull and format the MACE model to be used for our fine-tuning. 

We'll refit the MACE-OMAT-medium foundation model, which can be queried using the [MACE](https://github.com/ACEsuit/mace) Python interface; we can write this to an appropriate directory with a subprocess call. This is necessary in the current Notebook setup, because the default Python supports the Spack environment containing Flux, so we must set the Python executable to the environment's Python. 

In [ ]:
import torch

def query_model(model_name, generation):

    if generation == '1': # First generation

        from mace.calculators import mace_mp

        device = "cuda" if torch.cuda.is_available() else "cpu"

        # Load a passed foundation model
        model = mace_mp(
            model=model_name,  # or any listed foundation model
            device=device,
            return_raw_model=True
        )

        return model
    else:
        return None

model_name = 'medium-omat-0'
model_str = "model.model"

# Load a passed foundation model
model = query_model(model_name, generation)

if model: 
    mace_model_write_directory = os.path.join(ff_generation_path, model_name)
    os.makedirs(mace_model_write_directory, exist_ok=True)
    mace_model_write_path = os.path.join(mace_model_write_directory, model_str)
    torch.save(model, mace_model_write_path)
else: # No model queried, have already refit
    mace_model_write_path = glob(os.path.join(ff_generation_path, f'**/{model_str}'))[0]
    mace_model_write_directory = str(Path(mace_model_write_path).parent)
    model_name = Path(mace_model_write_path).parent.name

print(f'Model path: {mace_model_write_path}, model name: {model_name}')

We can now fit an ensemble of force fields using the executable provided in Ensemble-FF-Fit. 

In [ ]:
ff_fitting_cmd_args = ["mace_matensemble",
        "--inputs_directory", ff_inputs_path, 
        "--run_directory", mace_model_write_directory, 
        "--train_file", 'train.xyz', 
        "--test_file", 'test.xyz', 
        "--fits_per_runpath", '1', 
        "--foundation_model", "model.model", 
        "--finished_file", "*.model"]

ff_fitting_outputs_path = os.path.join(ff_path, 'outputs', f'generation_{generation}')
os.makedirs(ff_fitting_outputs_path, exist_ok=True)

dry_run_submitter(ff_fitting_cmd_args, ff_fitting_outputs_path, dry_run=False)

# B. Performing Uncertainty Quantification with the Force Field Ensemble

## 1. Run Melt/Quench Sampling Simulations

### a. Finite Temperature MD

Now we are at the main part of fine-tuning; selecting the new structures to add to the dataset and determining the best force field to spawn the next generation's ensemble. 

We will start by selecting structures from our VASP runs to be used a starting structures for finite-temperature MD runs. We will select structures from groups of parent structures (by mpid and calculation type) and adjust the sampling rates so that certain structures are more represented in our MD runs than others. 

In [ ]:
md_directory = os.path.abspath(os.path.join(os.getcwd(), 'MD'))
md_generation_path = os.path.join(md_directory, f'generation_{generation}')
md_FT_directory = os.path.join(md_generation_path, 'FT')
md_FT_inputs_directory = os.path.join(md_FT_directory, 'inputs_directory')
os.makedirs(md_FT_inputs_directory, exist_ok=True)

In [ ]:
import random
from collections import defaultdict
import re

def get_file_paths(paths, check_file='POSCAR'):
    all_paths = []
    all_parents = []
    for path in paths:
        for root, _, _ in os.walk(path):
            use_file = os.path.join(root, check_file)
            if os.path.exists(use_file):
                all_paths.append(root)
                all_parents.append(path)
    return all_paths, all_parents
     
def group_paths_by_mpid(paths, parents):
    groups = defaultdict(lambda: defaultdict(list))
    for p, parent in zip(paths, parents):
        for num in re.findall(r"mp-(\d+)", p):
            mpid = "mp-" + num
            groups[mpid][parent].append(p)
    return groups

def sample_paths(grouped_dct, mpids_of_interest, 
                 num_samples_gen, gen_rate, num_samples_interest, 
                 by_parent=True, seed=None):
    
    rng = random.Random(seed)
    nrng = np.random.default_rng(seed=seed)
    mpid_paths = []
    
    for mpid, mpid_dct in grouped_dct.items():
        if mpid in mpids_of_interest:
            num_samples = num_samples_interest
        else:
            num_samples = int(nrng.choice([0, num_samples_gen], 1,
              p=[1-gen_rate, gen_rate])[0])

        pooled = []
        for path_lsts in mpid_dct.values():
            if by_parent:
                sampled_paths = rng.sample(path_lsts, min(num_samples, len(path_lsts)))
                pooled.extend(sampled_paths)
            else:
                pooled.extend(path_lsts)
        
        mpid_sampled = rng.sample(pooled, min(num_samples, len(pooled)))
        mpid_paths += mpid_sampled 

    return mpid_paths

samp_root_paths, samp_parent_paths = get_file_paths([path_to_rescaled_poscars, path_to_defect_poscars, path_to_slab_dir, path_to_exfoliated_heteros], #path_to_uq_poscars], 
                            check_file='CONTCAR')
samp_root_paths_dct = group_paths_by_mpid(samp_root_paths, samp_parent_paths)

mpids_of_interest = list(substrate_layers_dct.keys()) + ['mp-541837', 'mp-23164']
randomly_sampled = sample_paths(samp_root_paths_dct, by_parent=True, 
                                mpids_of_interest=mpids_of_interest, 
                                num_samples_gen=1, gen_rate=0.4, num_samples_interest=2, seed=122 + int(generation))

for rs in randomly_sampled:
    print(rs)

## Now move to the inputs directory
sampled_poscar_exists = any(Path(md_FT_inputs_directory).rglob('POSCAR'))
if not sampled_poscar_exists:
    for sampled_path in randomly_sampled:
        subprocess.run(["submit_vaspflux",
                "--parent_directory", sampled_path,
                "--move", "--move_to", md_FT_inputs_directory], stdout=subprocess.DEVNULL)

We now have starting structures for our MD runs; we can write a Python script that uses the ASE package to run a finite-temperature MD protocol with MatEnsemble.

In [ ]:
ase_mace_ft_string = """import sys
import os
import torch
import gc
from mace.calculators import MACECalculator
from EnsembleFFFit.matensemble.lammps.helpers import parse_list
from ase.optimize import FIRE
from ase.filters import FrechetCellFilter
from ase.io import read
from ase.io import Trajectory
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from itertools import count
import ase.units as units
import json
import numpy as np
import random

def run_dynamics(atoms, timestep, temperature, friction, frequency, 
                 num_steps, to_attach, trajectory_path):
         
    dyn = Langevin(atoms=atoms, 
                   timestep=timestep, 
                   temperature_K=temperature, 
                   friction=friction)

    # attach the callback to run based on frequency
    dyn.attach(to_attach, frequency)

    # --- trajectory and attachments
    traj = Trajectory(trajectory_path, 'a', atoms)
    dyn.attach(traj.write, frequency, atoms)

    # Run the MD                   
    dyn.run(num_steps)

    # close trajectory file
    traj.close()
    
    return 

def temperature_scheduler(temp_value):

    if type(temp_value) == int:
        return temp_value
    elif type(temp_value) == tuple or type(temp_value) == list:
        options = np.linspace(temp_value[0], temp_value[1], temp_value[2])
        return float(random.sample(list(options), 1)[0]) 

if __name__ == "__main__":

    ff_list       = parse_list(sys.argv[1]) # Force field file paths
    input_list    = parse_list(sys.argv[2]) # Input file paths
    struct_list   = parse_list(sys.argv[3]) # Structure file paths
    output_list   = parse_list(sys.argv[4]) # LAMMPs output write paths

    n = len(ff_list)
    assert all(len(lst) == n for lst in (input_list, struct_list, output_list)), "All lists must be same length"

    for idx, (ff, inp, struct, output) in enumerate(zip(ff_list, input_list, struct_list, output_list), start=0):
        
        # 1a) Load the MACE model
        calculator = MACECalculator(model_path=ff, device="cuda" if torch.cuda.is_available() else "cpu")

        # 1b) Load the structure and configuration files
        with open(inp) as fh: 
            cfg = json.load(fh)
        init_conf = read(struct)
        init_conf.set_calculator(calculator)

        # 2) Initialize the calculation
        step_counter = count()
        trajectory_path = os.path.join(output, 'md_run.traj')
        if os.path.exists(trajectory_path): # Remove file if it exists
            os.remove(trajectory_path)

        # 3) Set the MD integration and property attachment
        dt = cfg['timestep'] * units.fs
        friction = cfg['friction'] / units.fs

        property_dict = {}

        def update_status():
            step = next(step_counter) * cfg['frequency']
            energy = init_conf.get_potential_energy()
            forces = init_conf.get_forces()
            property_dictionary = {
                'energy': float(energy),
                'fx': [float(f[0]) for f in forces],
                'fy': [float(f[1]) for f in forces],
                'fz': [float(f[2]) for f in forces],
            }
            property_dict[step] = property_dictionary

        # 4) Run the MD cycles
        for i in range(cfg.get("cycles", 1)):
        
            # Cell Minimization
            
            fcf = FrechetCellFilter(init_conf)
            opt = FIRE(fcf)
            opt.run(fmax=cfg['fmax'], steps=cfg['optsteps'])

            # high temperature
            
            ht = temperature_scheduler(cfg['high_temperature'])
            MaxwellBoltzmannDistribution(init_conf, temperature_K=ht)
            run_dynamics(init_conf, dt, ht, friction, cfg['frequency'], cfg['nsteps'], update_status, trajectory_path)

            # low temperature

            lt = temperature_scheduler(cfg['low_temperature'])
            run_dynamics(init_conf, dt, lt, friction, cfg['frequency'], cfg['nsteps'], update_status, trajectory_path)

        # --- after run: write the property_dict to disk once ---
        output_name = os.path.join(output, 'properties.json')
        with open(output_name, "w") as f:
            json.dump(property_dict, f, indent=4)

        # --- after run: free Python memory ---
        if torch.cuda.is_available():
            del calculator, init_conf   # remove large objects
            gc.collect()                # free Python memory
            torch.cuda.empty_cache()    # release unreferenced GPU memory back to CUDA driver"""

ase_mace_ft_path = os.path.join(md_FT_inputs_directory, 'ase_mace.py')
write_string(ase_mace_ft_path, ase_mace_ft_string)

This script requires a protocol json file to run the molecular dynamics.

In [ ]:
MD_json_path = os.path.join(md_FT_inputs_directory, 'in.mace.json')
cycles = 4

MD_dct = {
  "high_temperature": (2500, 5000, 6),
  "low_temperature": 300,
  "nsteps": 10000,
  "frequency": 250 * cycles,
  "friction": 0.05,
  "timestep": 0.5,
  "fmax": 0.05, 
  "optsteps": 200, 
  "cycles": cycles
}

write_json(MD_json_path, MD_dct)

Next, let's copy the default MACE model we used for our ensemble fine-tuning to a directory inside the FT MD directory. 

In [ ]:
md_FT_MACE_directory = os.path.join(md_FT_directory, model_name)
os.makedirs(md_FT_inputs_directory, exist_ok=True)
copy_files(mace_model_write_directory, md_FT_MACE_directory, patterns=("model.model"))

We can now launch the molecular dynamics simulations using our MatEnsemble call. 

In [ ]:
md_outputs_directory = os.path.join(md_directory, 'outputs')
md_ft_outputs_directory = os.path.join(md_outputs_directory, 'FT')
os.makedirs(md_ft_outputs_directory, exist_ok=True)
model_string = "model.model"

ft_md_cmd_args = ["lammps_matensemble",
                  "--run_directory", md_FT_MACE_directory,
                  "--inputs_directory", md_FT_inputs_directory,
                  "--ffield", model_string,
                  "--structure", "POSCAR",
                  "--control", "None",
                  "--in_lammps", "in.mace.json",
                  "--lammps_task", "ase_mace.py",
                  "--atoms_per_task", "1000",
                  "--lammps_task_order", "ffield", "in_lammps", "structure", 
                  "--parent_levels", "0",
                  "--gpus_per_task", "1",
                  "--cpus_per_task", "16",
                  "--atom_style", "atomic", 
                  "--finished_file", "properties.json"]

dry_run_submitter(ft_md_cmd_args, md_ft_outputs_directory, dry_run=False)

### b. Single Points

We will now create a directory where we run single point calculations using our ensemble of force fields for all structures generated from the MD simulations.  

In [64]:
md_SP_directory = os.path.join(md_generation_path, 'SP')
md_SP_inputs_directory = os.path.join(md_SP_directory, 'inputs_directory')
os.makedirs(md_SP_inputs_directory, exist_ok=True)

First, let's write POSCAR files from the different images along the MD trajectories. 

**Note:** We'll make sure that the directory naming is consistent between the FT and SP runs. Consistent naming conventions will come in handy later, when we are directly comparing between multiple MD runs and DFT ground truth properties. 

In [65]:
from ase.io.trajectory import Trajectory
from pymatgen.io.ase import AseAtomsAdaptor

def get_trajectory_paths(path, traj_name="md_run.traj"):
    all_paths = []
    for root, _, _ in os.walk(path):
        traj_path = os.path.join(root, "md_run.traj")
        if os.path.exists(traj_path):
            all_paths.append(traj_path)
    return all_paths

def write_trajectories_to_POSCARs(to_path, split_path, all_paths, cfg):
    aaa = AseAtomsAdaptor()
    
    for path in all_paths:
        base_path = path.split(split_path)[-1][1:] # Strip leading character
        write_to_path = os.path.dirname(os.path.join(to_path, base_path))
        os.makedirs(os.path.dirname(write_to_path), exist_ok=True)
        traj = Trajectory(path)
        for i, ase_atoms in enumerate(traj):
            structure = aaa.get_structure(ase_atoms)
            write_to_path_image = os.path.join(write_to_path, str(int(i * cfg['frequency'] * cfg['timestep']))) # By number of femtoseconds
            os.makedirs(write_to_path_image, exist_ok=True)
            structure.to(fmt='poscar', filename=os.path.join(write_to_path_image, 'POSCAR'))        
    return 

trajectory_paths = get_trajectory_paths(md_FT_MACE_directory)
write_trajectories_to_POSCARs(md_SP_inputs_directory, md_FT_MACE_directory, trajectory_paths, MD_dct)

Next, we'll write the input files to run single point calculations with ASE. 

In [66]:
ase_mace_sp_string = """import sys
import os
import torch
from mace.calculators import MACECalculator
from EnsembleFFFit.matensemble.lammps.helpers import parse_list
from ase.io import read, write
import json

if __name__ == "__main__":
    ff_list       = parse_list(sys.argv[1]) # Force field file paths
    struct_list   = parse_list(sys.argv[2]) # Structure file paths
    output_list   = parse_list(sys.argv[3]) # LAMMPs output write paths

    n = len(ff_list)
    assert all(len(lst) == n for lst in (struct_list, output_list)), "All lists must be same length"

    for idx, (ff, struct, output) in enumerate(zip(ff_list, struct_list, output_list), start=0):
        property_dictionary = {}

        # 1a) Load the MACE model
        calculator = MACECalculator(model_path=ff, device="cuda" if torch.cuda.is_available() else "cpu")

        # 1b) Load the structure file
        init_conf = read(struct)

        # 1c) Write the structure file
        write(filename=os.path.join(output, 'POSCAR'), images=init_conf)

        # 2) Compute the potential energy
        init_conf.set_calculator(calculator)
        energy = init_conf.get_potential_energy()
        forces = init_conf.get_forces()

        # 3) Write the energy to the output path
        property_dictionary['energy'] = energy
        property_dictionary['fx'] = [forces[i][0] for i in range(len(forces))]
        property_dictionary['fy'] = [forces[i][1] for i in range(len(forces))]
        property_dictionary['fz'] = [forces[i][2] for i in range(len(forces))]

        with open(os.path.join(output, 'properties.json'), "w") as f:
            json.dump(property_dictionary, f, indent=4)"""

ase_mace_sp_path = os.path.join(md_SP_inputs_directory, 'ase_mace.py')
write_string(ase_mace_sp_path, ase_mace_sp_string)

Finally, we'll copy the fitted force fields in a format so that MatEnsemble can run the collection of predictions. 

In [67]:
md_SP_MACE_directory = os.path.join(md_SP_directory, model_name)
os.makedirs(md_SP_inputs_directory, exist_ok=True)

# Then copy over all files
#ffield_pattern = '^MACE_([^._-]+)\\.model$' # Regex pattern for output MACE files
ffield_pattern = r"^MACE_(\d+)\.model$"

copied_ffs_exists = any(Path(md_SP_MACE_directory).rglob(model_string))
if not copied_ffs_exists:
    subprocess.run(["copy_by_pattern",
                    "--source_directory", mace_model_write_directory,
                    "--target_directory", md_SP_MACE_directory,
                    "--target_name", model_string, 
                    "--num_source_directory_directories", "4", 
                    "--pattern", ffield_pattern], stdout=subprocess.DEVNULL)

Now, we can submit the full workflow using MatEnsemble to compute single point energies. 

**Note**: We pass --parents_levels=1 to reduce latency time for the calculation. This groups the single point runs on the same GPU, instead of re-initializing a new single point for each calculation. 

In [68]:
md_outputs_sp_directory = os.path.join(md_outputs_directory, 'SP')
os.makedirs(md_outputs_sp_directory, exist_ok=True)

sp_md_cmd_args = ["lammps_matensemble",
                  "--run_directory", md_SP_MACE_directory,
                  "--inputs_directory", md_SP_inputs_directory,
                  "--ffield", model_string,
                  "--structure", "POSCAR",
                  "--control", "None",
                  "--in_lammps", "None",
                  "--lammps_task", "ase_mace.py",
                  "--atoms_per_task", "1000",
                  "--lammps_task_order", "ffield", "structure", 
                  "--parent_levels", "1",
                  "--gpus_per_task", "1",
                  "--cpus_per_task", "16",
                  "--atom_style", "atomic", 
                  "--finished_file", "properties.json"]

dry_run_submitter(sp_md_cmd_args, md_outputs_sp_directory, dry_run=False)

## 2. Determine the images with the greatest uncertainty

Now that we have run single points for the force field ensemble on the collection of structures, we can write some code that chooses the structures where force and energy variance is largest for the newest DFT runs.

We can do this in the same hacky way with some imports available in Ensemble-FF-Fit, but we will have to use subprocess to call a code snippet that can retrieve the sorted structures for us, based on variance scoring. 

**Note:** We need to specify with the ```label_tuple``` argument which parts of the path should contain the ```force_field``` label. All parts after (other than the last part of the path) specify the MD run, while the last part specifies the image. The ```valid_parsed_dct``` therefore looks like:

```
{force_field: 
  {md_run: 
    {image: 
      {energy: float, 
       structure: pmg.Structure, 
       fx: lst, 
       fy: lst, 
       fz: lst}
    }
  }
}
```
where the ```label_tuple``` corresponds to the ```md_run``` name. 

In [69]:
variance_code = r"""
import sys, json
from EnsembleFFFit.analysis.dict_parsers import ASEParser
from EnsembleFFFit.analysis.variance import get_structures_scores

path = sys.argv[1]
label_tuple = eval(sys.argv[2])
energy_weight = float(sys.argv[3])
force_weight = float(sys.argv[4])

valid_parser = ASEParser(path)
valid_parsed_dct = valid_parser.parse_directory(label_tuple)

labels, images, structures, scores = get_structures_scores(
    valid_parsed_dct,
    energy_weight,
    force_weight,
)

out = {
    "labels": labels,
    "images": images,
    "scores": scores,
    "structures": [s.as_dict() for s in structures],
}

print(json.dumps(out))
"""

uq_out = subprocess.check_output(
    [
        "/global/homes/r/rym/.conda/envs/unified/bin/python",
        "-c",
        variance_code,
        md_SP_MACE_directory,
        '(9, 15)', # label_tuple specified here
        '0.1', 
        '1.0'
    ],
    text=True
)

uq_data = json.loads(uq_out)

uq_labels = uq_data["labels"]
uq_images = uq_data["images"]
uq_scores = uq_data["scores"]
uq_structures = [Structure.from_dict(d) for d in uq_data["structures"]]

Let's look at the top structures with the largest variance across the ensemble:

In [70]:
used_labels = []
for i, uq_label in enumerate(uq_labels):
    if uq_label not in used_labels:
        print(f'Label: {uq_labels[i]}, Image: {uq_images[i]},  Score: {uq_scores[i]}')
        used_labels.append(uq_label)

Label: structures/defects/Al-O/Al5O8/mp-1182893/vacancy/vacancy3, Image: 22000,  Score: 0.9090367237416671
Label: structures/defects/Al-O/Al2O3/mp-1143/vacancy/vacancy0, Image: 33000,  Score: 0.844011290436533
Label: structures/heteros/exfoliated/mp-20485/2/mp-23164/1/2/spacing_0.6, Image: 26000,  Score: 0.7013182667448972
Label: structures/EoS/Al-O/Al2O3/mp-1938/volume_1.15, Image: 35500,  Score: 0.6528013190019082
Label: structures/EoS/Al-O/Al2O3/mp-7048/volume_1, Image: 0,  Score: 0.6523876144675484
Label: structures/slabs/Al-O/Al2O3/mp-1143/1, Image: 31500,  Score: 0.526042275873099
Label: structures/defects/Nb-Se/NbSe2/mp-1072113/interstitial/interstitial3, Image: 14000,  Score: 0.4811675869506853
Label: structures/EoS/Nb-Se/Nb2Se9/mp-541106/volume_0.85, Image: 11500,  Score: 0.3518851760013753
Label: structures/defects/Bi-Se/Bi8Se9/mp-1190284/interstitial/interstitial4, Image: 4000,  Score: 0.2607232969430415
Label: structures/defects/Nb-Se/Nb3Se4/mp-561/interstitial/interstitial

Most likely, the most poorly predicted structures come from the same starting structure(s). 

We (likely) don't want to bias our training dataset by including different images from these same trajectories, so instead let's select the single image with the largest variance from each individual run, and write these images to a new folder for DFT calculations. 

In [71]:
def select_structures(labels, images, structures, scores, total=80, 
                      score_cap=100, max_per_label=6, unique_run=True):
    if unique_run == True:
        count = 1
        selected_labels, selected_images, selected_structures, selected_scores = [], [], [], []
        for i, label in enumerate(labels):
            label_count = selected_labels.count(label)
            if label_count <= max_per_label and count <= total and scores[i] <= score_cap:
                selected_labels.append(label)
                selected_images.append(images[i])
                selected_structures.append(structures[i])
                selected_scores.append(scores[i])
                count += 1
        return selected_labels, selected_images, selected_structures, selected_scores
    else:
        return labels[:total], images[:total], structures[:total], scores[:total]

def write_new_structures(to_dir, labels, images, structures):
    for i, structure in enumerate(structures):
        write_path = os.path.join(to_dir, os.path.join(labels[i], str(images[i])))
        os.makedirs(write_path, exist_ok=True)
        structure.to(
            fmt="poscar",
            filename=os.path.join(write_path, "POSCAR"),
        )
    return 

In [72]:
sel_labels, sel_images, sel_structures, sel_scores = select_structures(uq_labels, uq_images, uq_structures, uq_scores)
path_to_model_variance = os.path.join(path_to_uq_poscars, f'generation_{generation}')
write_new_structures(path_to_model_variance, sel_labels, sel_images, sel_structures)

## 3. Calculate DFT single-point energies

We can now regenerate single-point inputs for these structures and run them with DFT. 

In [73]:
subprocess.run(["generate_vaspflux",
                         "--poscars_directory", path_to_model_variance,
                         "--vasp_yaml", sp_yaml_path,
                         "--vasp_submission", perlmutter_vasp_submission_path, 
                          "--atoms_per_node", "100"], stdout=subprocess.DEVNULL)

variance_dft_cmd_args = ["matsemble_vaspflux",
                     "--parent_directory", 
                     path_to_model_variance]

dft_variance_outputs_path = os.path.join(dft_outputs_path, 'uq_variance')
os.makedirs(dft_variance_outputs_path, exist_ok=True)

dry_run_submitter(variance_dft_cmd_args, dft_variance_outputs_path, dry_run=False)

<string>:31: BadInputSetWarning: Overriding the POTCAR functional is generally not recommended  as it significantly affects the results of calculations and compatibility with other calculations done with the same input set. Note that some POTCAR symbols specified in the configuration file may not be available in the selected functional.


## 4. Determine the best force field


Now that we have new data to add to our training, our final step is to select the "best" force field from our ensemble to use as the starting force field for the next generation. 

We will use our DFT validation data, and then compare energy and force predictions across the ensemble. 

**Note:** Choosing the validation data will depend upon the application at hand, and should be selected carefully. 

### a. MD SP runs with Ensemble

Having converged the images from our AIMD validation set, we can now run FF predictions based on our Ensemble from this collection of structures. Let's first construct a Validation directory, with the appropriate inputs, to be used to run our collection of single points. 

In [74]:
# Create the directory 
md_Validation_directory = os.path.join(md_generation_path, 'Validation')
md_Validation_inputs_directory = os.path.join(md_Validation_directory, 'inputs_directory')
os.makedirs(md_Validation_inputs_directory, exist_ok=True)

# Create the outputs directory
md_Validation_outputs_directory = os.path.join(md_outputs_directory, 'Validation')
os.makedirs(md_Validation_outputs_directory, exist_ok=True)

# Write the appropriate files to the inputs_directory
ase_mace_Validation_path = os.path.join(md_Validation_inputs_directory, 'ase_mace.py')
write_string(ase_mace_Validation_path, ase_mace_sp_string)

# Move converged single-point structures to inputs_directory
copied_poscars_valid_exists = any(Path(md_Validation_inputs_directory).rglob('POSCAR'))
if not copied_poscars_valid_exists:
    subprocess.run(["submit_vaspflux",
                "--parent_directory", path_to_aimd_sps,
                "--move", "--move_to", md_Validation_inputs_directory], stdout=subprocess.DEVNULL)

In [75]:
# Copy over the force fields
md_Validation_MACE_directory = os.path.join(md_Validation_directory, model_name)
os.makedirs(md_Validation_MACE_directory, exist_ok=True)

# Ensemble with proper formatting
copied_ffs_valid_exists = any(Path(md_Validation_MACE_directory).rglob(model_string))
if not copied_ffs_valid_exists:
    subprocess.run(["copy_by_pattern",
                    "--source_directory", mace_model_write_directory,
                    "--target_directory", md_Validation_MACE_directory,
                    "--target_name", 'model.model', 
                    "--num_source_directory_directories", "4", 
                    "--pattern", ffield_pattern], stdout=subprocess.DEVNULL)

# Original with proper formatting
original_force_field_directory = os.path.join(md_Validation_directory, 'original')
os.makedirs(original_force_field_directory, exist_ok=True)
copy_files(mace_model_write_directory, original_force_field_directory, patterns=('model.model'))

In [76]:
# Finally, submit the single-point runs
valid_md_cmd_args = ["lammps_matensemble",
                  "--run_directory", md_Validation_directory,
                  "--inputs_directory", md_Validation_inputs_directory,
                  "--ffield", model_string,
                  "--structure", "POSCAR",
                  "--control", "None",
                  "--in_lammps", "None",
                  "--lammps_task", "ase_mace.py",
                  "--atoms_per_task", "1000",
                  "--lammps_task_order", "ffield", "structure", 
                  "--parent_levels", "1",
                  "--gpus_per_task", "1",
                  "--cpus_per_task", "16",
                  "--atom_style", "atomic",
                  "--finished_file", "properties.json"]

dry_run_submitter(valid_md_cmd_args, md_Validation_outputs_directory, dry_run=False)

Let's parse these single points into a formatted dictionary, like we did previously with the other single points. We will write another short function to accomplish this through a subprocess call. 

**Note:** Once again, it is important to make sure that your tuples are correct for parsing, so that the same structures can be compared across the different MD runs and DFT ground truths. 

**Note:** Also note that the relative force and energy weighting will influence which force field is selected. In this example, we more heavily weight force agreement. 

In [77]:
validation_code = r"""
import sys, json
from EnsembleFFFit.analysis.dict_parsers import ASEParser, VASPParser
from EnsembleFFFit.analysis.best_force_field import rank_ff_scores

md_path = sys.argv[1]
dft_path = sys.argv[2]
orig_path = sys.argv[3]

md_tuple = eval(sys.argv[4])
dft_tuple = eval(sys.argv[5])
orig_tuple = eval(sys.argv[6])

energy_weight = float(sys.argv[7])
force_weight = float(sys.argv[8])

md_parser = ASEParser(md_path)
md_parsed_dct = md_parser.parse_directory(md_tuple)

orig_parser = ASEParser(orig_path)
orig_parsed_dct = orig_parser.parse_directory(orig_tuple)

md_parsed_dct.update(orig_parsed_dct)

dft_parser = VASPParser(dft_path)
dft_parsed_dct = dft_parser.parse_directory(dft_tuple)

labels, scores = rank_ff_scores(
    md_parsed_dct,
    dft_parsed_dct,
    energy_weight,
    force_weight,
)

out = {
    "labels": labels,
    "scores": scores,
}

print(json.dumps(out))
"""

valid_out = subprocess.check_output(
    [
        "/global/homes/r/rym/.conda/envs/unified/bin/python",
        "-c",
        validation_code,
        md_Validation_MACE_directory,
        path_to_aimd_sps,
        original_force_field_directory, 
        '(10, 15)',
        '(6, 7)', 
        '(9, 10)',
        '0.1', 
        '10.0'
    ],
    text=True
)

validation_data = json.loads(valid_out)

validation_labels = validation_data["labels"]
validation_scores = validation_data["scores"]

Let's inspect the best performers. 

In [78]:
best_index = 0
valid_number = 20
best_model_dir = os.path.join(md_Validation_MACE_directory, validation_labels[best_index])
validation_labels[:valid_number], validation_scores[:valid_number]

(['UQ2_EoS/freeze_7/energy_1/0/0',
  'UQ2_UQ1_EoS/freeze_7/energy_1/0/0',
  'UQ1_EoS/freeze_7/energy_1/0/0',
  'UQ1_EoS/freeze_7/energy_2/0/0',
  'UQ2_UQ1_EoS/freeze_7/energy_2/0/0',
  'UQ2_EoS/freeze_7/energy_2/0/0',
  'UQ1_EoS/freeze_7/energy_3/0/0',
  'original',
  'UQ2_UQ1_EoS/freeze_7/energy_3/0/0',
  'UQ2_EoS/freeze_7/energy_3/0/0',
  'Slab_UQ1_EoS/freeze_7/energy_3/0/0',
  'Slab_Exfoliated_UQ2_UQ1_EoS/freeze_7/energy_3/0/0',
  'Slab_Exfoliated_UQ2_EoS/freeze_7/energy_3/0/0',
  'Slab_UQ2_UQ1_EoS/freeze_7/energy_3/0/0',
  'Slab_Exfoliated_UQ1_EoS/freeze_7/energy_3/0/0',
  'Slab_UQ2_EoS/freeze_7/energy_3/0/0',
  'Slab_EoS/freeze_7/energy_3/0/0',
  'Slab_Exfoliated_EoS/freeze_7/energy_3/0/0',
  'UQ1_EoS/freeze_6/energy_3/0/0',
  'UQ2_EoS/freeze_6/energy_1/0/0'],
 [3.7127676785413564,
  3.712767678541358,
  3.800513709105604,
  3.8089041747071577,
  3.810820664793039,
  3.8108206647930416,
  3.8394134670896354,
  3.862471370388598,
  3.8926114005691277,
  3.892611400569128,
  3.97280

### b. Compare Best to DFT for UQ Structures

Let's explore which of the UQ structures are the worst-predicted by our force field. 

In [79]:
best_comparison_code = r"""
import sys, json
from EnsembleFFFit.analysis.dict_parsers import ASEParser, VASPParser
from EnsembleFFFit.analysis.variance import get_structures_scores

ff_path = sys.argv[1]
dft_path = sys.argv[2]

ff_label_tuple = eval(sys.argv[3])
dft_label_tuple = eval(sys.argv[4])

energy_weight = float(sys.argv[5])
force_weight = float(sys.argv[6])

ff_parser = ASEParser(ff_path)
ff_parsed_dct = ff_parser.parse_directory(ff_label_tuple)

dft_parser = VASPParser(dft_path)
dft_parsed_dct = dft_parser.parse_directory(dft_label_tuple)

ff_map = {list(ff_parsed_dct.keys())[0]: list(dft_parsed_dct.keys())[0]}

filtered_md_dct = {
    ff: {
        md: {
            img: ff_parsed_dct[ff][md][img]
            for img in dft_parsed_dct[ff_map[ff]][md]
        }
        for md in dft_parsed_dct[ff_map[ff]]
    }
    for ff in ff_parsed_dct
}

filtered_md_dct.update(dft_parsed_dct)

labels, images, structures, scores = get_structures_scores(
    filtered_md_dct,
    energy_weight,
    force_weight,
)

out = {
    "labels": labels,
    "images": images,
    "scores": scores,
    "structures": [s.as_dict() for s in structures],
}

print(json.dumps(out))
"""

best_uq_directory = os.path.join(md_SP_MACE_directory, validation_labels[best_index])
best_out = subprocess.check_output(
    [
        "/global/homes/r/rym/.conda/envs/unified/bin/python",
        "-c",
        best_comparison_code,
        best_uq_directory,
        path_to_model_variance,
        '(10, 15)', # label_tuple specified here
        '(6, 10)', 
        '0.1', 
        '10.0'
    ],
    text=True
)

best_data = json.loads(best_out)

best_labels = best_data["labels"]
best_images = best_data["images"]
best_scores = best_data["scores"]
best_structures = [Structure.from_dict(d) for d in best_data["structures"]]

In [80]:
used_best_labels = []
for i, best_label in enumerate(best_labels):
    if best_label not in used_best_labels:
        print(f'Label: {best_label}, Image: {best_images[i]},  Score: {best_scores[i]}')
        used_best_labels.append(best_label)

Label: structures/defects/Nb-Se/NbSe2/mp-2207/vacancy/vacancy1, Image: 12500,  Score: 0.8521175341463809
Label: structures/defects/Nb-Se/NbSe2/mp-1072113/interstitial/interstitial3, Image: 14000,  Score: 0.7715870399658515
Label: structures/EoS/Nb-Se/Nb2Se9/mp-541106/volume_0.85, Image: 11500,  Score: 0.7554338671475807
Label: structures/defects/Nb-Se/Nb3Se4/mp-561/interstitial/interstitial3, Image: 33000,  Score: 0.47659223009698526
Label: structures/defects/Al-O/Al2O3/mp-1143/vacancy/vacancy0, Image: 33000,  Score: 0.4357777133099357
Label: structures/EoS/Al-O/Al2O3/mp-1938/volume_1.15, Image: 35500,  Score: 0.24017467313233293
Label: structures/EoS/Bi-Se/Bi2Se3/mp-23164/volume_0.85, Image: 26500,  Score: 0.20694267866692315
Label: structures/heteros/exfoliated/mp-20485/2/mp-23164/1/2/spacing_0.6, Image: 33500,  Score: 0.2043024731520462
Label: structures/defects/Bi-Se/Bi8Se9/mp-1190284/interstitial/interstitial4, Image: 4000,  Score: 0.16678851167985598
Label: structures/defects/Bi-

### c. Write Best force field as the start to the next generation

Having now ranked our force fields and inspected the performance of our most recent addition on the UQ data, we can select the next force field for fitting, and restart the fitting cycle. 

In [81]:
best_force_field_name = validation_labels[0].replace('/', '_')
next_generation_ff_dir = os.path.join(ff_path, f'generation_{str(int(generation) + 1)}', best_force_field_name)
os.makedirs(next_generation_ff_dir, exist_ok=True)
copy_files(best_model_dir, next_generation_ff_dir, patterns=('model.model'))